In [0]:
dbutils.widgets.text("taxi_type","yellow","Taxi type")
dbutils.widgets.text("year","2025","Year")
dbutils.widgets.text("month","1","Month")
dbutils.widgets.dropdown(
    "force_reload",
    "false",
    ["false", "true"],
    "Force reload"
)

In [0]:
raw_taxi_type = dbutils.widgets.get("taxi_type")
raw_year = dbutils.widgets.get("year")
raw_month = dbutils.widgets.get("month")
raw_force_reload = dbutils.widgets.get("force_reload")

taxi_type = raw_taxi_type.lower().strip()
year = int(raw_year)
month = int(raw_month)
force_reload_text = raw_force_reload.strip().lower()

if force_reload_text not in {"true", "false"}:
    raise ValueError(
        "force_reload must be either 'true' or 'false'"
    )

force_reload = force_reload_text == "true"
print(f"Taxi type: {taxi_type}")
print(f"Year: {year}")
print(f"Month: {month}")
print(f"Force reload: {force_reload}")



In [0]:
from datetime import datetime, timezone

now_utc = datetime.now(timezone.utc)
current_year = now_utc.year
current_month = now_utc.month

if taxi_type not in {"yellow", "green"}:
    raise ValueError(
        f"Unsupported taxi_type: {taxi_type}. "
        "Allowed values: yellow, green."
    )

if year < 2009:
    raise ValueError("Year cannot be earlier than 2009.")

if not 1 <= month <= 12:
    raise ValueError("Month must be between 1 and 12.")

if (year, month) > (current_year, current_month):
    raise ValueError(
        f"Requested period {year}-{month:02d} is in the future."
    )

print("Parameter validation passed.")

In [0]:
from uuid import uuid4
from datetime import datetime, timezone


run_id = uuid4().hex
started_at = datetime.now(timezone.utc)

completed_at = None
http_status = None
bytes_downloaded = 0
row_count = None
error_message = None

period = f"{year}-{month:02d}"

file_name = f"{taxi_type}_tripdata_{period}.parquet"
source_url = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    f"{file_name}"
)

base_volume_path = "/Volumes/workspace/urban_mobility_bronze/landing"

destination_directory = (
    f"{base_volume_path}/{taxi_type}/"
    f"year={year}/month={month:02d}"
)
destination_path = f"{destination_directory}/{file_name}"


temporary_directory = f"{base_volume_path}/_temporary/{run_id}"
temporary_path = f"{temporary_directory}/{file_name}"

print(f"Run ID: {run_id}")
print(f"Started at: {started_at.isoformat()}")
print(f"Source URL: {source_url}")
print(f"Destination: {destination_path}")
print(f"Temporary path: {temporary_path}")

In [0]:
from pathlib import Path
from datetime import datetime, timezone

destination_file = Path(destination_path)
destination_exists = destination_file.is_file()

should_download = force_reload or not destination_exists
completed_at = None

if destination_exists and not force_reload:
    ingestion_status = "SKIPPED_ALREADY_EXISTS"
    should_download = False
    completed_at = datetime.now(timezone.utc)

    print("File already exists. Download skipped.")

else:
    ingestion_status = "READY_TO_DOWNLOAD"

    print(f"File will be downloaded from: {source_url}")

print(f"Should download: {should_download}")
print(f"Ingestion status: {ingestion_status}")
print(f"Completed at: {completed_at}")

In [0]:
import requests
from pathlib import Path

bytes_downloaded = 0
expected_bytes = None
http_status = None

if should_download:
    Path(temporary_directory).mkdir(
        parents=True,
        exist_ok=True
    )

    try:
        with requests.get(
            source_url,
            stream=True,
            timeout=(10, 300)
        ) as response:
            http_status = response.status_code
            response.raise_for_status()

            content_length = response.headers.get("Content-Length")

            if content_length is not None:
                expected_bytes = int(content_length)

            with open(temporary_path, "wb") as temporary_file:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        temporary_file.write(chunk)
                        bytes_downloaded += len(chunk)

        if bytes_downloaded == 0:
            raise ValueError(
                "Download completed but the file is empty."
            )

        if (
            expected_bytes is not None
            and bytes_downloaded != expected_bytes
        ):
            raise ValueError(
                "Downloaded size does not match Content-Length. "
                f"Expected {expected_bytes}, got {bytes_downloaded}."
            )

        ingestion_status = "DOWNLOADED_TO_TEMP"

    except Exception:
        ingestion_status = "DOWNLOAD_FAILED"

        temporary_file_path = Path(temporary_path)

        if temporary_file_path.exists():
            temporary_file_path.unlink()

        raise

print(f"HTTP status: {http_status}")
print(f"Downloaded bytes: {bytes_downloaded:,}")
print(f"Downloaded MB: {bytes_downloaded / 1024 / 1024:.2f}")
print(f"Expected bytes: {expected_bytes}")
print(f"Ingestion status: {ingestion_status}")

In [0]:
from pathlib import Path

validation_passed = False
row_count = None
file_size_bytes = None
missing_columns = []

required_columns = {
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "total_amount"
}

if should_download:
    temporary_file = Path(temporary_path)

    try:
        # 1. Fișierul trebuie să existe
        if not temporary_file.is_file():
            raise FileNotFoundError(
                f"Temporary file does not exist: {temporary_path}"
            )

        # 2. Fișierul nu trebuie să fie gol
        file_size_bytes = temporary_file.stat().st_size

        if file_size_bytes == 0:
            raise ValueError("Temporary file is empty.")

        # 3. Spark trebuie să-l poată deschide ca Parquet
        candidate_df = spark.read.parquet(temporary_path)

        # 4. Verificăm coloanele obligatorii
        missing_columns = sorted(
            required_columns - set(candidate_df.columns)
        )

        if missing_columns:
            raise ValueError(
                f"Missing required columns: {missing_columns}"
            )

        # 5. Datasetul trebuie să conțină rânduri
        row_count = candidate_df.count()

        if row_count == 0:
            raise ValueError(
                "Parquet file is valid but contains no rows."
            )

        validation_passed = True
        ingestion_status = "VALIDATED"

    except Exception:
        validation_passed = False
        ingestion_status = "VALIDATION_FAILED"
        raise

    finally:
        print(f"File size: {file_size_bytes}")
        print(f"Row count: {row_count}")
        print(f"Missing columns: {missing_columns}")
        print(f"Validation passed: {validation_passed}")
        print(f"Ingestion status: {ingestion_status}")

else:
    print("Validation skipped because no download was required.")

In [0]:
import os
from pathlib import Path

if should_download:
    if not validation_passed:
        raise RuntimeError(
            "Cannot promote a file that failed validation."
        )

    temporary_file = Path(temporary_path)
    destination_file = Path(destination_path)

    destination_file.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # Protecție pentru cazul în care altă rulare a creat
    # fișierul după verificarea inițială.
    if destination_file.exists() and not force_reload:
        temporary_file.unlink(missing_ok=True)
        ingestion_status = "SKIPPED_ALREADY_EXISTS"

        print(
            "Another execution already created the destination file."
        )

    else:
        # Fișierele sunt în același managed volume.
        # os.replace mută fișierul și permite overwrite
        # atunci când force_reload este activ.
        os.replace(
            temporary_path,
            destination_path
        )

        if not destination_file.is_file():
            raise RuntimeError(
                "Promotion completed without creating the destination."
            )

        if temporary_file.exists():
            raise RuntimeError(
                "Temporary file still exists after promotion."
            )

        ingestion_status = "PROMOTED"

        print("File promoted successfully.")
        print(f"Final destination: {destination_path}")

else:
    print("Promotion skipped because no download was required.")

print(f"Final ingestion status: {ingestion_status}")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.urban_mobility_ops.ingestion_runs (
    run_id STRING NOT NULL,
    taxi_type STRING NOT NULL,
    data_year INT NOT NULL,
    data_month INT NOT NULL,
    source_url STRING NOT NULL,
    destination_path STRING NOT NULL,
    started_at TIMESTAMP NOT NULL,
    completed_at TIMESTAMP NOT NULL,
    status STRING NOT NULL,
    http_status INT,
    bytes_downloaded BIGINT,
    row_count BIGINT,
    force_reload BOOLEAN NOT NULL,
    error_message STRING
)
USING DELTA
COMMENT 'Audit history for urban mobility source-file ingestion';

In [0]:
from datetime import datetime, timezone
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    BooleanType,
    TimestampType
)

audit_table = "workspace.urban_mobility_ops.ingestion_runs"

# Finalizăm rularea dacă completed_at nu a fost deja
# setat în cazul SKIPPED_ALREADY_EXISTS.
if completed_at is None:
    completed_at = datetime.now(timezone.utc)

if completed_at < started_at:
    raise ValueError(
        "Invalid audit timestamps: completed_at is earlier than started_at. "
        "Run the notebook again from the first cell."
    )
# Unele valori nu există în cazul unui skip.
# De exemplu, nu avem HTTP status dacă nu am făcut download.
audit_http_status = globals().get("http_status")
audit_bytes_downloaded = globals().get("bytes_downloaded")
audit_row_count = globals().get("row_count")
audit_error_message = globals().get("error_message")

audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("taxi_type", StringType(), False),
    StructField("data_year", IntegerType(), False),
    StructField("data_month", IntegerType(), False),
    StructField("source_url", StringType(), False),
    StructField("destination_path", StringType(), False),
    StructField("force_reload", BooleanType(), False),
    StructField("status", StringType(), False),
    StructField("http_status", IntegerType(), True),
    StructField("bytes_downloaded", LongType(), True),
    StructField("row_count", LongType(), True),
    StructField("started_at", TimestampType(), False),
    StructField("completed_at", TimestampType(), False),
    StructField("error_message", StringType(), True)
])

audit_data = [(
    run_id,
    taxi_type,
    int(year),
    int(month),
    source_url,
    destination_path,
    force_reload,
    ingestion_status,
    audit_http_status,
    audit_bytes_downloaded,
    audit_row_count,
    started_at,
    completed_at,
    audit_error_message
)]

audit_df = spark.createDataFrame(
    audit_data,
    schema=audit_schema
)

display(audit_df)

In [0]:
%sql
SELECT
    run_id,
    taxi_type,
    data_year,
    data_month,
    status,
    http_status,
    bytes_downloaded,
    row_count,
    started_at,
    completed_at
FROM workspace.urban_mobility_ops.ingestion_runs
ORDER BY started_at DESC;

In [0]:
%sql
SELECT
    run_id,
    status,
    started_at,
    completed_at,
    completed_at >= started_at AS timestamps_valid
FROM workspace.urban_mobility_ops.ingestion_runs;